# FCIdump: Integrating Psi4 into PySCF in a Reproducable Way

In [61]:
import numpy as np
import psi4 
import os
import pyscf
from pyscf import ao2mo, tools, gto
import pyscf.mcscf
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Generate a Psi4 Molecule and run FCIdump

In [62]:
xyz_path = os.path.join(os.path.expanduser("~"), "DDLUCJ", "check_amplitudes", "diatomics", "HLi.xyz")

with open(xyz_path, 'r') as f:
    xyz_text = f.read()

qmol = psi4.qcdb.Molecule.from_string(xyz_text, dtype='xyz')
mol = psi4.geometry(qmol.create_psi4_string_from_molecule() + "\nsymmetry c1\n")

psi4.core.clean()
psi4.core.be_quiet()

psi4.set_options({
    'basis': 'STO-3G',
    # 'scf_type': 'pk',
    'reference': 'rhf',
    'e_convergence': 1e-8,
    'd_convergence': 1e-8,
    'print_mos': True,
    'frozen_DOCC': [0]
})

rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
# FCIDUMP
psi4.driver.p4util.fcidump(scf_wfn, fname="FF_INTDUMP")

# Step 2: Generate Same molecule in PySCF, and save for future use

In [63]:
# Specify molecule properties
open_shell = False
spin_sq = 0
 
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=xyz_path,
    basis="STO-3G",
    symmetry="c1",
)
 
# Define active space
n_frozen = 0
active_space = range(n_frozen, mol.nao_nr())
 
# Get molecular integrals
scf_ = pyscf.scf.RHF(mol).run()
num_orbitals = len(active_space)
n_electrons = int(sum(scf_.mo_occ[active_space]))
num_elec_a = (n_electrons + mol.spin) // 2
num_elec_b = (n_electrons - mol.spin) // 2
cas = pyscf.mcscf.CASCI(scf_, num_orbitals, (num_elec_a, num_elec_b))
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)

converged SCF energy = -7.86219514126873


# Step 3: Reformat FCIdump files and run PySCF from FCIdump

In [64]:
def fix_fcidump_header(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Identify header lines between &FCI and &END
    header_start = None
    header_end = None
    for i, line in enumerate(lines):
        if '&FCI' in line:
            header_start = i
        if '&END' in line:
            header_end = i
            break

    if header_start is None or header_end is None:
        raise ValueError("Could not find a valid &FCI ... &END block")

    # Join and sanitize header
    header_lines = lines[header_start:header_end+1]
    header_joined = ' '.join(line.strip() for line in header_lines)
    header_joined = header_joined.replace('\n', '').replace('  ', ' ').replace(' ,', ',')

    # Replace old header with single-line version
    new_lines = lines[:header_start] + [header_joined + '\n'] + lines[header_end+1:]

    # Write back to file
    with open(filename, 'w', encoding='utf-8') as f:
        f.writelines(new_lines)

    print(f"[OK] Fixed header in {filename}")


In [65]:
fix_fcidump_header('FF_INTDUMP')

[OK] Fixed header in FF_INTDUMP


In [66]:
scf_psi4 = scf.from_fcidump('FF_INTDUMP',False).run()

Parsing FF_INTDUMP
converged SCF energy = -7.86219514124685


/home/amousso3/miniconda3/envs/ddlucj-env/lib/python3.12/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute energy_nuc because it is not JSON-serializable
  warnings.warn(msg)
/home/amousso3/miniconda3/envs/ddlucj-env/lib/python3.12/site-packages/pyscf/gto/mole.py:1293: UserWarning: Function mol.dumps drops attribute intor_symmetric because it is not JSON-serializable
  warnings.warn(msg)


# Step 4: Check Difference Between Mol obhjects

In [67]:
num_orbitals_psi4 = len(active_space)
n_electrons_psi4 = int(sum(scf_psi4.mo_occ[active_space]))
num_elec_a_psi4 = (n_electrons + mol.spin) // 2
num_elec_b_psi4 = (n_electrons - mol.spin) // 2
cas_psi4 = pyscf.mcscf.CASCI(scf_psi4, num_orbitals, (num_elec_a, num_elec_b))
mo_psi4 = cas.sort_mo(active_space, base=0)
hcore_psi4, nuclear_repulsion_energy_psi4 = cas.get_h1cas(mo)
eri_psi4 = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)

In [68]:
df = pd.DataFrame(columns=['Number of Orbitals', 'Number of Electrons',
                           'MO Coefficients', 'One Electron', 
                           "Nuclear Repulsion", "Two Electron"])


In [69]:
df.loc[len(df)] = [num_orbitals_psi4 - num_orbitals, n_electrons_psi4 - n_electrons, mo_psi4 - mo, hcore_psi4 -hcore, nuclear_repulsion_energy_psi4 - nuclear_repulsion_energy,eri_psi4 - eri ]

In [70]:
df

,Number of Orbitals,Number of Electrons,MO Coefficients,One Electron,Nuclear Repulsion,Two Electron
0,0,0,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0....","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0....",0.0,"[[[[0. 0. 0. 0. 0. 0.], [0. 0. 0. 0. 0. 0.], [..."


In [71]:
np.isclose(eri_psi4,eri)

array([[[[ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True]],

        [[ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True]],

        [[ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True],
         [ True,  True,  True,  True,  True,  True]],

        [[ True,  True,  True,  True,  T

# Check FCIdump from PySCF file

In [72]:
from pyscf.tools import fcidump

